<a href="https://colab.research.google.com/github/Suhani145/Suhani145/blob/main/FakeNewsProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup & Libraries

##Install packages

In [41]:
# 1️⃣ Install required libraries
!pip install datasets gradio scikit-learn pandas torch --quiet
!pip uninstall -y transformers
!pip install transformers datasets accelerate


Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)


In [1]:
import transformers
print(transformers.__version__)   # Should be something like 4.46.1 or higher

5.0.0


In [13]:
!pip show transformers

Name: transformers
Version: 5.2.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: peft, sentence-transformers


In [14]:
import transformers
print(transformers.__file__)   # Shows the file path of the imported module

/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


## Import libraries

In [2]:
# 2️⃣ Imports
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    default_data_collator
)
import gradio as gr
from google.colab import drive

# Load & Explore Data

##Mount Google Drive

In [3]:
# 3️⃣ Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


##Load CSVs from Google Drive

In [4]:
# 4️⃣ Load CSVs from Google Drive
# Replace the paths below with the exact folder where your CSVs are
true_data = pd.read_csv('/content/drive/MyDrive/FakeNewsProject/True.csv')
fake_data = pd.read_csv('/content/drive/MyDrive/FakeNewsProject/Fake.csv')

##Prepare Data

In [5]:
true_data['Target'] = 1  # True
fake_data['Target'] = 0  # Fake

data = pd.concat([true_data, fake_data], ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data['combined'] = data['title'] + " " + data['text']

print("Dataset shape:", data.shape)
print("Class distribution:\n", data['Target'].value_counts())

Dataset shape: (44898, 6)
Class distribution:
 Target
0    23481
1    21417
Name: count, dtype: int64


In [6]:
# 5️⃣ Train / val / test split
X = data['combined']
y = data['Target']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 31428, Val size: 6735, Test size: 6735


# Baseline Model: TF-IDF + Logistic Regression

In [7]:
# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    data['combined'], data['Target'], test_size=0.3, random_state=42, stratify=data['Target']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Predictions
y_val_pred = lr_model.predict(X_val_tfidf)
y_test_pred = lr_model.predict(X_test_tfidf)

# Evaluation
print("Validation Accuracy:", (y_val_pred==y_val).mean())
print("\nTest Classification Report:\n", classification_report(y_test, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

# Misclassified examples
misclassified = X_test[(y_test != y_test_pred)]
misclassified_df = pd.DataFrame({
    'Text': misclassified,
    'True Label': y_test[(y_test != y_test_pred)],
    'Predicted': y_test_pred[(y_test != y_test_pred)]
})
print("\nSome misclassified examples:\n", misclassified_df.head(5))

Validation Accuracy: 0.984112843355605

Test Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      3523
           1       0.99      0.99      0.99      3212

    accuracy                           0.99      6735
   macro avg       0.99      0.99      0.99      6735
weighted avg       0.99      0.99      0.99      6735

Confusion Matrix:
 [[3482   41]
 [  25 3187]]

Some misclassified examples:
                                                     Text  True Label  \
8510   BREAKING: Iran Publicly Humiliates Obama…Unvei...           0   
41090  “G#d d*mn America”: DISTURBING PHOTOS Illustra...           0   
4696   SAY WHAT? PROSECUTORS Claim Convicted ISLAMIC ...           0   
3682   HOLY CONTRACEPTION! Pope Francis Tells Latin A...           0   
8081   Obama speaks up for protester but is derided b...           1   

       Predicted  
8510           1  
41090          1  
4696           1  
3682           1  

# Transformer Model: DistilBERT

## Tokenization

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
MAX_LEN = 128

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors=None)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
train_enc = tokenize(list(X_train))
val_enc = tokenize(list(X_val))
test_enc = tokenize(list(X_test))

# PyTorch Dataset
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = NewsDataset(train_enc, list(y_train))
val_dataset = NewsDataset(val_enc, list(y_val))
test_dataset = NewsDataset(test_enc, list(y_test))

## Model initialization

In [10]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Trainer & training arguments

In [21]:
# import torch
# from transformers import default_data_collator

# def custom_data_collator(batch):
#     """
#     batch: list of tuples (input_dict, label)
#     """
#     # Separate inputs and labels
#     inputs = [item[0] for item in batch]
#     labels = [item[1] for item in batch]

#     # Collate inputs (padding, etc.) using the default collator
#     batch_inputs = default_data_collator(inputs)

#     # Stack labels (each label is a 0‑dim tensor)
#     batch_inputs['labels'] = torch.stack(labels)

#     return batch_inputs

In [11]:
# Training Arguments
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/FakeNewsProject/model_results',  # save to Drive
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir='/content/drive/MyDrive/FakeNewsProject/logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
)

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


##Train the model

In [25]:
sample = train_dataset[0]
print(sample)
print(type(sample))

({'input_ids': tensor([  101,  3537,  5347,  9555,  8398,  2044,  6703, 22867,  1006,  2678,
         1007,  3951,  2392, 23195,  6221,  8398,  2003,  1999,  1996,  2739,
         2153,  2005,  4808,  2076,  2028,  1997,  2010, 22867,  1999,  2358,
         1012,  3434,  1998,  2010,  3190,  8320,  2001,  3844,  2091, 13463,
         3036,  5936,  1012,  1996,  3537,  5347,  2018,  8401,  2616,  2005,
         1996, 22301,  2006,  5958,  3944,  2044,  2739,  3631,  1997,  1996,
        17783,  1012,  1996,  4808,  1999,  2358,  1012,  3434,  2187, 13337,
        18618,  1998,  6703,  2044, 10958, 17062,  8398,  5470, 24916,  2787,
         2008,  5128,  2037,  2398,  2006,  2111,  2040,  2079,  2025,  5993,
         2007,  2068,  2001,  6413,  1012,  2023,  2234,  2074,  2028,  2154,
         2044,  1037,  2158,  2001, 26476,  1011, 11696,  2011,  1037,  8398,
        10129,  2012,  1037,  8320,  1999,  2167,  3792,  1012,  2044,  1996,
         2358,  1012,  8840,  4173,  8320,  1010,

In [12]:
# Train DistilBERT
trainer.train()
trainer.save_model('/content/drive/MyDrive/FakeNewsProject/model_results')
tokenizer.save_pretrained('/content/drive/MyDrive/FakeNewsProject/model_results')

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000055,0.001924,0.999703,1.000000,0.999378,0.999689
2,0.000018,0.001702,0.999703,0.999689,0.999689,0.999689


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/FakeNewsProject/model_results/tokenizer_config.json',
 '/content/drive/MyDrive/FakeNewsProject/model_results/tokenizer.json')

## Evaluate on test set

In [14]:
# Evaluate on Test Set
preds_output = trainer.predict(test_dataset)
y_test_pred_bert = np.argmax(preds_output.predictions, axis=1)

print("Test Classification Report (DistilBERT):\n", classification_report(list(y_test), y_test_pred_bert))
print("Confusion Matrix:\n", confusion_matrix(list(y_test), y_test_pred_bert))

Test Classification Report (DistilBERT):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3523
           1       1.00      1.00      1.00      3212

    accuracy                           1.00      6735
   macro avg       1.00      1.00      1.00      6735
weighted avg       1.00      1.00      1.00      6735

Confusion Matrix:
 [[3523    0]
 [   2 3210]]


In [16]:
# Optional: compare with baseline side-by-side

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("\n" + "="*50)
print("Comparison on Test Set:")
print("{:<20} {:<10} {:<10} {:<10} {:<10}".format("Model","Accuracy","Precision","Recall","F1"))
print("{:<20} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
    "Logistic Regression",
    accuracy_score(y_test, y_test_pred),
    precision_score(y_test, y_test_pred),
    recall_score(y_test, y_test_pred),
    f1_score(y_test, y_test_pred)
))
print("{:<20} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
    "DistilBERT",
    accuracy_score(y_test, y_test_pred_bert),
    precision_score(y_test, y_test_pred_bert),
    recall_score(y_test, y_test_pred_bert),
    f1_score(y_test, y_test_pred_bert)
))


Comparison on Test Set:
Model                Accuracy   Precision  Recall     F1        
Logistic Regression  0.9902     0.9873     0.9922     0.9898    
DistilBERT           0.9997     1.0000     0.9994     0.9997    


# Interactive UI

# Gradio interface for testing any headline + text

In [13]:
print("\n" + "="*50)
print("Launching Gradio interface...")

clickbait_words = ['shocking', 'amazing', 'secret', 'breaking', 'simple trick', 'reveals', 'hidden', 'you won’t believe']

def predict_fake_news(headlines, clickbait_boost=True):
    # Ensure headlines is a list
    if isinstance(headlines, str):
        headlines = [headlines]
    # Tokenize
    inputs = tokenizer(headlines, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
    # Move to same device as model
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

    trainer.model.eval()
    with torch.no_grad():
        logits = trainer.model(**inputs).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

    results = []
    for headline, pred, prob in zip(headlines, preds, probs):
        label = 'True' if pred == 1 else 'Fake'
        confidence = prob[pred] * 100

        # Clickbait boost (small adjustment)
        if clickbait_boost and any(word.lower() in headline.lower() for word in clickbait_words):
            if label == 'Fake':
                confidence = min(confidence + 5, 99)
            else:
                confidence = max(confidence - 5, 1)

        results.append(f"[{label} - {confidence:.1f}%] {headline}")
    return "\n".join(results)  # return a single string for Gradio

# Example test
example_headlines = [
    "Shocking Secret Billionaire Reveals How to Become Rich Overnight",
    "Scientists Discover New Method to Reverse Climate Change",
    "Politician Caught Lying About Tax Records in Explosive Leak"
]
print("\nSample predictions:")
print(predict_fake_news(example_headlines))

iface = gr.Interface(
    fn=predict_fake_news,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter news headlines here, one per line...", label="News Headlines"),
        gr.Checkbox(value=True, label="Enable Clickbait Boost")
    ],
    outputs=gr.Textbox(label="Predictions"),
    title="Fake News Detection with Clickbait Boost",
    description="Enter headlines to classify as True or Fake. Clickbait words slightly increase Fake probability."
)

iface.launch(share=True)


Launching Gradio interface...

Sample predictions:
[Fake - 99.0%] Shocking Secret Billionaire Reveals How to Become Rich Overnight
[Fake - 94.7%] Scientists Discover New Method to Reverse Climate Change
[Fake - 100.0%] Politician Caught Lying About Tax Records in Explosive Leak
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://561c528f2e20daedb7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
